# 📘 Day 36 — Random Forest: Hyperparameter Tuning

**Author:** Sahil-K-Y 
**Phase:** 3 — Tree Models & SVM 
**Date:** Day 036 of 214-Day AI/ML Roadmap

---
## 📌 Overview

Today we focus on **systematically tuning** Random Forest hyperparameters using **RandomizedSearchCV** and **GridSearchCV** to squeeze maximum performance.

### 🎯 Learning Objectives
- Master **RandomizedSearchCV** for efficient hyperparameter search
- Understand when to use **Randomized vs Grid** search
- Tune `max_depth`, `min_samples_split`, `min_samples_leaf`, `criterion`
- Learn to **interpret search results** and avoid overfitting to validation
- Build **learning curves** to diagnose model behavior

---
## 🧠 1. RandomizedSearchCV vs GridSearchCV

| Aspect | GridSearchCV | RandomizedSearchCV |
|--------|-------------|--------------------|
| **Search** | All combinations | Random subset |
| **Time** | Exponential growth | Fixed budget (n_iter) |
| **Best for** | Small param space (<100 combos) | Large param space |
| **Guarantees** | Finds best in grid | May miss best, but usually close |

### When to Use Which:
- **GridSearchCV:** 2-3 parameters with few values each
- **RandomizedSearchCV:** 4+ parameters or continuous distributions
- **Strategy:** Start with Randomized (broad search) → refine with Grid (narrow search)

---
## 🧠 2. Parameter Distributions for RandomizedSearch

```python
from scipy.stats import randint, uniform

param_distributions = {
    'n_estimators': randint(50, 500),        # uniform integer [50, 500)
    'max_depth': [5, 10, 15, 20, 30, None],  # discrete choices
    'min_samples_split': randint(2, 20),      # uniform integer [2, 20)
    'min_samples_leaf': randint(1, 10),       # uniform integer [1, 10)
    'max_features': ['sqrt', 'log2', 0.3, 0.5, 0.7],
    'criterion': ['gini', 'entropy'],
    'bootstrap': [True],
}
```

### Key Tips:
- Use `randint` for integer ranges (samples uniformly)
- Use `uniform` for float ranges
- Use lists for discrete choices
- Set `n_iter=100` for a good balance of speed and coverage

---
## 🧠 3. Learning Curves — Diagnosing Bias vs Variance

### What Learning Curves Show:
- **X-axis:** Number of training samples
- **Y-axis:** Score (accuracy/error) on train AND validation sets

### Interpretation:
| Pattern | Diagnosis | Fix |
|---------|-----------|-----|
| Train high, Val low, **big gap** | **High Variance** (overfitting) | More data, regularize, fewer features |
| Train low, Val low, **small gap** | **High Bias** (underfitting) | More features, less regularization, complex model |
| Train high, Val high, **small gap** | **Good fit** | Ship it! |

### For Random Forest:
- RF rarely has high bias (deep trees are flexible)
- Main risk is **variance** when n_estimators is too low
- Learning curves help find **minimum training data needed**

---
## 🧠 4. Post-Tuning Validation Strategy

### ⚠️ Common Mistake: Overfitting to Validation Set
If you run hundreds of search iterations, you might **overfit to the CV folds**.

### Solution: Nested Cross-Validation
```
Outer Loop (5 folds): Estimates true generalization performance
    └── Inner Loop (3 folds): Tunes hyperparameters
```

### Practical Alternative:
1. Use RandomizedSearch on 80% of data (train + val)
2. Report performance on held-out 20% (test set)
3. Never look at test set during tuning

---
## 💻 Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     StratifiedKFold, RandomizedSearchCV,
                                     GridSearchCV, learning_curve)
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (classification_report, accuracy_score,
                             ConfusionMatrixDisplay)
from sklearn.datasets import fetch_openml
from scipy.stats import randint, uniform

import warnings
warnings.filterwarnings('ignore')

plt.style.use('ggplot')
sns.set_palette('Set2')
print('✅ All libraries imported successfully!')

---
## 📊 Dataset: Steel Plates Fault Detection

The **Steel Plates Faults** dataset contains **1941 steel plate samples** with **27 features** (luminosity, geometry, shape indices) classified into **7 fault types** (Pastry, Z-Scratch, K-Scratch, Stains, Dirtiness, Bumps, Other).

**1941 samples, 27 features, 7 classes**

**Why this dataset?** A real **industrial quality control** problem with enough complexity to benefit from hyperparameter tuning. Multiple fault types with varying frequencies make it a great tuning challenge.

In [ ]:
# --- Load Steel Plates Fault Dataset ---
steel = fetch_openml('steel-plates-fault', version=3, as_frame=True, parser='auto')
df = steel.frame

target_col = steel.target_names[0] if steel.target_names else df.columns[-1]
X = df.drop(columns=[target_col]).astype(float)
y = LabelEncoder().fit_transform(df[target_col])

print(f'Shape: {X.shape}')
print(f'Number of classes: {len(np.unique(y))}')
print(f'Class distribution:')
print(pd.Series(y).value_counts().sort_index())
print(f'\nFeature names: {list(X.columns)}')
df.head()

---
## ✏️ Exercise 1: Baseline Random Forest
Train a default RF as baseline.

**Tasks:**
1. Split 80/20 stratified
2. Train RF with default params
3. 5-fold CV accuracy
4. Print OOB score
5. This is your baseline to beat

In [ ]:
# YOUR CODE HERE


---
## ✏️ Exercise 2: RandomizedSearchCV
Broad search across many hyperparameters.

**Tasks:**
1. Define param_distributions (use randint/uniform for continuous)
2. Run RandomizedSearchCV with n_iter=100, cv=5
3. Print best params and best score
4. Show top 10 parameter combinations
5. How much improvement over baseline?

In [ ]:
# YOUR CODE HERE


---
## ✏️ Exercise 3: GridSearchCV Refinement
Narrow search around best params from Randomized.

**Tasks:**
1. Define narrow grid around best params (±20%)
2. Run GridSearchCV with cv=5
3. Compare with RandomizedSearch results
4. Is the improvement significant?

In [ ]:
# YOUR CODE HERE


---
## ✏️ Exercise 4: Learning Curves
Diagnose bias vs variance with learning curves.

**Tasks:**
1. Plot learning curve for the tuned model
2. Use `sklearn.model_selection.learning_curve`
3. Plot training size vs train/val accuracy
4. Diagnose: is the model overfitting or underfitting?
5. How much training data is "enough"?

In [ ]:
# YOUR CODE HERE


---
## ✏️ Exercise 5: Final Test Evaluation
Evaluate the tuned model on the held-out test set.

**Tasks:**
1. Train best model on full training set
2. Predict on test set
3. Classification report with per-class metrics
4. Confusion matrix visualization
5. Which fault types are hardest to detect?

In [ ]:
# YOUR CODE HERE


---
## 📝 Key Takeaways
1. Baseline vs tuned improvement: ...
2. Most impactful hyperparameter: ...
3. RandomizedSearch vs GridSearch winner: ...
4. Bias vs Variance diagnosis: ...